In [1]:
%pip install pandas odfpy openpyxl statsmodels scikit-learn prophet matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# SETTINGS AND IMPORTS
# ============================================================
import os, json, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 150,
                     'axes.grid': True, 'grid.alpha': 0.3})

DATA_DIR = '.'          # folder containing the .ods files
OUT_DIR = os.path.expanduser('~/Downloads')      # folder for all generated files
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)

FILES = {
    'PORT0201': f'{DATA_DIR}/port0201.ods',
    'PORT0301': f'{DATA_DIR}/port0301.ods',
    'PORT0502': f'{DATA_DIR}/port0502.ods',
}
FOCUS_PORTS = ['Dover', 'Felixstowe', 'Southampton']  # case-study ports
BREXIT_YEAR = 2021   # end of transition period: 1 January 2021
COVID_YEAR = 2020    # acute pandemic shock year

def num(s):
    """Convert a column to numeric, coercing errors to NaN."""
    return pd.to_numeric(s, errors='coerce')

In [3]:
# ---- Download the six DfT tables from GOV.UK ----
import urllib.request

URLS = {
    'port0201.ods': 'https://assets.publishing.service.gov.uk/media/6a622055abcde513b38b6de8/port0201.ods',
    'port0301.ods': 'https://assets.publishing.service.gov.uk/media/6a6220bfe2d191f0bc8b6dfb/port0301.ods',
    'port0502.ods': 'https://assets.publishing.service.gov.uk/media/6a26c86356960b0542c0b19a/port0502.ods',
}

for filename, url in URLS.items():
    if not os.path.exists(filename):
        print('Downloading', filename, '...')
        urllib.request.urlretrieve(url, filename)
    print(filename, '-', os.path.getsize(filename), 'bytes')


port0201.ods - 284041 bytes
port0301.ods - 2569525 bytes
port0502.ods - 246234 bytes


In [4]:
# ============================================================
# PHẦN 1: DATA UNDERSTANDING - SỐ LIỆU BAN ĐẦU
# ============================================================
print("="*50)
print("PHẦN 1: DATA UNDERSTANDING - KHÁM PHÁ DỮ LIỆU THÔ")
print("="*50)

# Đọc sheet 'Data' (dòng tiêu đề nằm ở index 3) và sheet 'Tonnage_(Both_Directions)' (dòng tiêu đề nằm ở index 6)
p0301_raw = pd.read_excel(FILES['PORT0301'], sheet_name='Data', engine='odf', skiprows=3, header=0)
p0201_raw = pd.read_excel(FILES['PORT0201'], sheet_name='Data', engine='odf', skiprows=3, header=0)
p0502_raw = pd.read_excel(FILES['PORT0502'], sheet_name='Tonnage_(Both_Directions)', engine='odf', skiprows=6, header=0)

print(f"PORT0301 thô: {p0301_raw.shape[0]} quan sát")
print(f"PORT0201 thô: {p0201_raw.shape[0]} quan sát")
print(f"PORT0502 thô: {p0502_raw.shape[0]} cảng (trên {p0502_raw.shape[1]} cột thời gian)")

PHẦN 1: DATA UNDERSTANDING - KHÁM PHÁ DỮ LIỆU THÔ
PORT0301 thô: 48420 quan sát
PORT0201 thô: 4463 quan sát
PORT0502 thô: 54 cảng (trên 74 cột thời gian)


In [5]:
print(p0201_raw.columns.tolist())
print(p0201_raw.head(3))

['Year', 'CargoCode', 'Direction', 'Tonnage', 'Units', 'Region', 'TEU', 'Dummy']
   Year  CargoCode        Direction      Tonnage  Units         Region  TEU  \
0  2000         11  Both Directions  2374.687515    0.0       Domestic  NaN   
1  2000         11  Both Directions  4115.306580    0.0  International  NaN   
2  2000         11  Both Directions     9.740905    0.0    Unspecified  NaN   

   Dummy  
0      1  
1      1  
2      1  


In [6]:
# PHẦN 2: DATA PREPARATION - LÀM SẠCH & LỌC (2009-2025)
# ============================================================
print("\n" + "="*50)
print("PHẦN 2: DATA PREPARATION - SỐ LIỆU SAU KHI LÀM SẠCH")
print("="*50)

# --- Xử lý PORT0301 ---
# (Lưu ý: Sheet Data của 0301 dùng 'Cargo Group Name')
p0301_clean = p0301_raw.dropna(subset=['Cargo Group Name']).copy()
p0301_clean['Tonnage'] = num(p0301_clean['Tonnage'])
p0301_clean = p0301_clean.dropna(subset=['Tonnage'])
p0301_clean = p0301_clean[(p0301_clean['Year'] >= 2009) & (p0301_clean['Year'] <= 2025)]

# --- Xử lý PORT0201 ---
# (Lưu ý: Sheet Data của 0201 dùng 'CargoCode')
p0201_clean = p0201_raw.dropna(subset=['CargoCode']).copy()
p0201_clean['Tonnage'] = num(p0201_clean['Tonnage'])
p0201_clean = p0201_clean.dropna(subset=['Tonnage'])
p0201_clean = p0201_clean[(p0201_clean['Year'] >= 2009) & (p0201_clean['Year'] <= 2025)]

# --- Xử lý PORT0502 ---
def prep_0502_all_ports(df, direction_label):
    df_clean = df.dropna(subset=['Major Port']).copy()
    
    # Chỉ lấy các cột Quý (chứa chữ Q)
    quarter_cols = [col for col in df_clean.columns if 'Q' in str(col) and 'Percentage' not in str(col) and 'Four quarter' not in str(col)]
    
    # Lấy TẤT CẢ các cảng, không lọc qua FOCUS_PORTS nữa
    df_focus = df_clean[['Major Port'] + quarter_cols]
    
    # Xoay dữ liệu dọc (Melt từ Wide -> Long)
    df_long = pd.melt(df_focus, id_vars=['Major Port'], value_vars=quarter_cols, 
                      var_name='Quarter', value_name='Tonnage')
    df_long['Direction'] = direction_label
    df_long['Tonnage'] = num(df_long['Tonnage'])
    df_long = df_long.dropna(subset=['Tonnage'])
    
    # Tách lấy năm để áp dụng điều kiện lọc
    df_long['Quarter_Clean'] = df_long['Quarter'].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    df_long['Year'] = df_long['Quarter_Clean'].str[:4].astype(int)
    df_long = df_long[(df_long['Year'] >= 2009) & (df_long['Year'] <= 2025)]
    
    # Chuẩn hóa cột Quarter sang định dạng DateTime
    df_long['Date'] = pd.to_datetime(df_long['Quarter_Clean'].str[:4] + '-' + 
                                     (df_long['Quarter_Clean'].str[-1].astype(int) * 3 - 2).astype(str).str.zfill(2) + '-01')
    return df_long.drop(columns=['Year', 'Quarter_Clean'])

p0502_clean = prep_0502_all_ports(p0502_raw, 'Both Directions')
p0502_clean = p0502_clean[p0502_clean['Major Port'] != 'Total at all major ports'].copy()

# ============================================================
# PHẦN 3: BÁO CÁO KẾT QUẢ ĐẦU RA 
# ============================================================
print(f"PORT0301 đã làm sạch: {p0301_clean.shape[0]} quan sát (từ {p0301_clean['Year'].min()} - {p0301_clean['Year'].max()})")
print(f"PORT0201 đã làm sạch: {p0201_clean.shape[0]} quan sát (từ {p0201_clean['Year'].min()} - {p0201_clean['Year'].max()})")
print(f"PORT0502 đã làm sạch: {p0502_clean.shape[0]} quan sát (bao gồm toàn bộ {p0502_clean['Major Port'].nunique()} cảng)")



PHẦN 2: DATA PREPARATION - SỐ LIỆU SAU KHI LÀM SẠCH
PORT0301 đã làm sạch: 28513 quan sát (từ 2009 - 2025)
PORT0201 đã làm sạch: 2725 quan sát (từ 2009 - 2025)
PORT0502 đã làm sạch: 3604 quan sát (bao gồm toàn bộ 53 cảng)


In [7]:
import os
import pandas as pd

# ============================================================
# CẤU HÌNH THƯ MỤC
# ============================================================
DATA_DIR = '.'
# Tự động trỏ đường dẫn lưu file về thư mục Downloads của máy tính
OUT_DIR = os.path.expanduser('~/Downloads')

# Đảm bảo sử dụng đúng file .ods đã tải về
FILES = {
    'PORT0201': f'{DATA_DIR}/port0201.ods',
    'PORT0301': f'{DATA_DIR}/port0301.ods',
    'PORT0502': f'{DATA_DIR}/port0502.ods',
}

def num(s):
    return pd.to_numeric(s, errors='coerce')

# ============================================================
# 1. TRÍCH XUẤT VÀ LÀM SẠCH DỮ LIỆU TỪ FILE .ODS GỐC
# ============================================================

# --- PORT0301 ---
p0301_raw = pd.read_excel(FILES['PORT0301'], sheet_name='Data', engine='odf', skiprows=3, header=0)
p0301_clean = p0301_raw.dropna(subset=['Cargo Group Name']).copy()
p0301_clean['Tonnage'] = num(p0301_clean['Tonnage'])
p0301_clean = p0301_clean.dropna(subset=['Tonnage'])
p0301_clean = p0301_clean[(p0301_clean['Year'] >= 2009) & (p0301_clean['Year'] <= 2025)]

# --- PORT0201 ---
p0201_raw = pd.read_excel(FILES['PORT0201'], sheet_name='Data', engine='odf', skiprows=3, header=0)
p0201_clean = p0201_raw.dropna(subset=['CargoCode']).copy()
p0201_clean['Tonnage'] = num(p0201_clean['Tonnage'])
p0201_clean = p0201_clean.dropna(subset=['Tonnage'])
p0201_clean = p0201_clean[(p0201_clean['Year'] >= 2009) & (p0201_clean['Year'] <= 2025)]

# --- PORT0502 ---
p0502_raw = pd.read_excel(FILES['PORT0502'], sheet_name='Tonnage_(Both_Directions)', engine='odf', skiprows=6, header=0)

def prep_0502_all_ports(df):
    df_clean = df.dropna(subset=['Major Port']).copy()
    
    quarter_cols = [col for col in df_clean.columns if 'Q' in str(col) and 'Percentage' not in str(col) and 'Four quarter' not in str(col)]
    df_focus = df_clean[['Major Port'] + quarter_cols]
    
    df_long = pd.melt(df_focus, id_vars=['Major Port'], value_vars=quarter_cols, 
                      var_name='Quarter', value_name='Tonnage')
    df_long['Direction'] = 'Both Directions'
    df_long['Tonnage'] = num(df_long['Tonnage'])
    df_long = df_long.dropna(subset=['Tonnage'])
    
    df_long['Quarter_Clean'] = df_long['Quarter'].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()
    df_long['Year'] = df_long['Quarter_Clean'].str[:4].astype(int)
    df_long = df_long[(df_long['Year'] >= 2009) & (df_long['Year'] <= 2025)]
    
    df_long['Date'] = pd.to_datetime(df_long['Quarter_Clean'].str[:4] + '-' + 
                                     (df_long['Quarter_Clean'].str[-1].astype(int) * 3 - 2).astype(str).str.zfill(2) + '-01')
    return df_long.drop(columns=['Year', 'Quarter_Clean'])

p0502_clean = prep_0502_all_ports(p0502_raw)
p0502_clean = p0502_clean[p0502_clean['Major Port'] != 'Total at all major ports'].copy()   # ← THÊM DÒNG NÀY

# Cell thứ hai - CHỈ xuất Excel, dùng dữ liệu đã có từ cell thứ nhất
import os
OUT_DIR = os.path.expanduser('~/Downloads')
output_path = f'{OUT_DIR}/Cleaned_Maritime_Data_2009_2025.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    p0301_clean.to_excel(writer, sheet_name='PORT0301', index=False)
    p0201_clean.to_excel(writer, sheet_name='PORT0201', index=False)
    p0502_clean.to_excel(writer, sheet_name='PORT0502_Both', index=False)
print("Đã lưu:", output_path)
print("Số cảng PORT0502:", p0502_clean['Major Port'].nunique())  # phải 53

Đã lưu: /Users/rachelnguyen/Downloads/Cleaned_Maritime_Data_2009_2025.xlsx
Số cảng PORT0502: 53


In [8]:
# ============================================================
# TẠO DATAFRAME 'PANEL' TỪ PORT0301 ĐỂ CHUẨN BỊ PHÂN TÍCH
# ============================================================
# Tạo bản sao từ dữ liệu PORT0301 đã làm sạch
panel = p0301_clean.copy()

# Đổi tên cột cho khớp với code của bạn
panel = panel.rename(columns={'Port Name': 'PortName'})
panel = panel[panel['Direction'] == 'Both Directions'].copy()

# Hàm phân loại Mode (Lấy Ro-Ro và Lo-Lo từ cột Cargo Group Name)
def get_mode(cargo_name):
    if 'Ro-Ro' in str(cargo_name):
        return 'Ro-Ro'
    elif 'Lo-Lo' in str(cargo_name):
        return 'Lo-Lo'
    else:
        return 'Other'

panel['Mode'] = panel['Cargo Group Name'].apply(get_mode)

# Chỉ giữ lại hàng Ro-Ro và Lo-Lo
panel = panel[panel['Mode'] != 'Other']
panel = panel.groupby(['Year', 'PortName', 'Mode'], as_index=False)['Tonnage'].sum()

# =================
# ============================================================
# DESCRIPTIVE STATISTICS
# ============================================================

# --- Định nghĩa focus (3 cảng case study) ---
focus = panel[panel['PortName'].isin(FOCUS_PORTS)].copy()

# --- Bảng summary statistics (toàn bộ cảng, 2015-2024) ---
window = panel[(panel['Year'] >= 2015) & (panel['Year'] <= 2024)]
summary = window.groupby('Mode')['Tonnage'].agg(
    N='count', mean='mean', SD='std',
    median='median', min='min', max='max').round(1)
print('Summary statistics (2015-2024):')
print(summary)
summary.to_csv(f'{OUT_DIR}/summary_statistics.csv')

# --- Bảng trước/sau Brexit (quốc gia) ---
print('\nPre vs Post-Brexit by mode:')
for mode in ['Ro-Ro', 'Lo-Lo']:
    md = panel[panel['Mode'] == mode]
    # Trung bình tổng sản lượng mỗi năm giai đoạn 2015-2019
    pre = md[md['Year'].between(2015, 2019)].groupby('Year')['Tonnage'].sum().mean()
    # Trung bình tổng sản lượng mỗi năm giai đoạn 2021-2024
    post = md[md['Year'].between(2021, 2024)].groupby('Year')['Tonnage'].sum().mean()
    print(f'  {mode}: pre={pre:.1f}  post={post:.1f}  change={(post/pre-1)*100:+.1f}%')

# --- % thay đổi trước/sau Brexit theo từng cảng (Route Diversion) ---
changes = []
for mode in ['Ro-Ro', 'Lo-Lo']:
    s = panel[panel['Mode'] == mode]
    pre = s[s['Year'].between(2015, 2019)].groupby('PortName')['Tonnage'].mean()
    post = s[s['Year'].between(2021, 2024)].groupby('PortName')['Tonnage'].mean()
    c = pd.DataFrame({'Mode': mode, 'pre_2015_19': pre, 'post_2021_24': post})
    c['pct_change'] = (c['post_2021_24'] / c['pre_2015_19'] - 1) * 100
    changes.append(c.reset_index())

mode_change = pd.concat(changes).round(1)
mode_change.to_csv(f'{OUT_DIR}/mode_change_by_port.csv', index=False)

# --- In các cảng Ro-Ro giảm/tăng mạnh nhất ---
rr = mode_change[mode_change['Mode'] == 'Ro-Ro'].dropna().sort_values('pct_change')
print('\nLargest Ro-Ro declines and gains:')
print(pd.concat([rr.head(6), rr.tail(6)])[['PortName', 'pct_change']].to_string(index=False))

Summary statistics (2015-2024):
         N    mean      SD  median  min      max
Mode                                            
Lo-Lo  197  3188.5  5469.0   912.2  0.0  25346.3
Ro-Ro  286  3589.4  5022.6  2054.9  0.1  27084.3

Pre vs Post-Brexit by mode:
  Ro-Ro: pre=105824.7  post=99418.0  change=-6.1%
  Lo-Lo: pre=65558.9  post=59745.6  change=-8.9%

Largest Ro-Ro declines and gains:
         PortName  pct_change
         Plymouth       -74.6
         Ramsgate       -71.8
Tees & Hartlepool       -40.5
             Hull       -40.4
         Newhaven       -40.3
             Tyne       -38.1
          Heysham        12.9
      Warrenpoint        13.2
          Belfast        18.1
           London        20.7
            Larne        35.0
        Cairnryan        35.0


In [9]:
pre_years  = [2015, 2016, 2017, 2018, 2019]
post_years = [2021, 2022, 2023, 2024]

pre = (panel[panel['Year'].isin(pre_years)]
       .groupby(['PortName','Mode'], as_index=False)['Tonnage'].mean()
       .rename(columns={'Tonnage':'pre_mean'}))

post = (panel[panel['Year'].isin(post_years)]
        .groupby(['PortName','Mode'], as_index=False)['Tonnage'].mean()
        .rename(columns={'Tonnage':'post_mean'}))

base = pre.merge(post, on=['PortName','Mode'], how='inner')
base['pct_change'] = (base['post_mean'] / base['pre_mean'] - 1) * 100

# Lo-Lo, xep theo QUY MO chu khong theo pct_change
ll = base[base['Mode'] == 'Lo-Lo'].sort_values('pre_mean', ascending=False).copy()
ll['share_pre_pct'] = ll['pre_mean'] / ll['pre_mean'].sum() * 100

print("Lo-Lo xep theo quy mo san luong truoc Brexit:")
print(ll[['PortName','pre_mean','post_mean','pct_change','share_pre_pct']]
      .round(1).head(15).to_string(index=False))

ll.round(2).to_csv(f'{OUT_DIR}/lolo_route_diversion.csv', index=False)

Lo-Lo xep theo quy mo san luong truoc Brexit:
           PortName  pre_mean  post_mean  pct_change  share_pre_pct
         Felixstowe   24205.1    18615.3       -23.1           37.0
             London   11979.7    14281.8        19.2           18.3
        Southampton    9925.1     8458.2       -14.8           15.2
          Liverpool    5589.8     5318.9        -4.8            8.5
  Tees & Hartlepool    2445.5     2361.6        -3.4            3.7
              Forth    2273.7     2118.3        -6.8            3.5
Grimsby & Immingham    2045.4     2146.6         4.9            3.1
               Hull    1775.9     2058.5        15.9            2.7
            Belfast    1688.6     1750.1         3.6            2.6
             Medway     930.0      484.9       -47.9            1.4
            Bristol     885.7      980.0        10.6            1.4
              Clyde     624.3      506.7       -18.8            1.0
               Tyne     404.0      295.4       -26.9            0.6
  

In [10]:
import numpy as np

route_map = (mode_change
             .dropna(subset=['pct_change'])
             .rename(columns={'pre_2015_19': 'pre_mean',
                              'post_2021_24': 'post_mean'})
             .copy())

# ty trong cua cang trong tong san luong cua CHINH mode do
route_map['share_pre_pct'] = (route_map['pre_mean'] /
                              route_map.groupby('Mode')['pre_mean'].transform('sum') * 100)

route_map['direction'] = np.where(route_map['pct_change'] > 0, 'Increase', 'Decrease')
route_map['abs_pct']   = route_map['pct_change'].abs()

route_map = route_map.sort_values(['Mode', 'pct_change']).round(2)
route_map = route_map[['PortName', 'Mode', 'pre_mean', 'post_mean',
                       'pct_change', 'abs_pct', 'direction', 'share_pre_pct']]

route_map.to_csv(f'{OUT_DIR}/route_map_combined.csv', index=False, encoding='utf-8-sig')

print(route_map.groupby('Mode').size().to_string())
print(f"\nDa luu: {OUT_DIR}/route_map_combined.csv  ({len(route_map)} dong)")
route_map.head()
mc = pd.read_csv(f'{OUT_DIR}/mode_change_by_port.csv')
route_map = mc.dropna(subset=['pct_change']).rename(
    columns={'pre_2015_19': 'pre_mean', 'post_2021_24': 'post_mean'}).copy()
# roi chay tiep tu dong share_pre_pct o tren

Mode
Lo-Lo    19
Ro-Ro    28

Da luu: /Users/rachelnguyen/Downloads/route_map_combined.csv  (47 dong)


In [11]:
# ============================================================
# REGRESSION 
# ============================================================

def report(model, label):
    """Print effect sizes (percent) for the key regression terms."""
    print(f'--- {label} | R2={model.rsquared:.3f} n={int(model.nobs)} ---')
    for t in ['post_brexit', 'post_brexit:roro', 'covid']:
        if t in model.params:
            b, p_ = model.params[t], model.pvalues[t]
            print(f'  {t:>18}: {100*(np.exp(b)-1):+.1f}%  (p={p_:.4f})')

# ------------------------------------------------------------
# BƯỚC 1: TẠO CÁC BIẾN (VARIABLES) TRÊN TỆP PANEL TRƯỚC
# ------------------------------------------------------------
panel['cell'] = panel['PortName'] + '_' + panel['Mode']   # tạo cell
panel = panel[panel['Tonnage'] > 0].copy()   # rồi mới bỏ 0
panel['log_tonnage'] = np.log(panel['Tonnage'])   # log bình thường, không +1
panel['roro'] = (panel['Mode'] == 'Ro-Ro').astype(int)
panel['post_brexit'] = (panel['Year'] >= BREXIT_YEAR).astype(int)
panel['covid'] = (panel['Year'] == COVID_YEAR).astype(int)
panel['trend'] = panel['Year'] - 2000

# Tạo tệp focus từ panel đã có đầy đủ biến
focus = panel[panel['PortName'].isin(FOCUS_PORTS)].copy()

# ------------------------------------------------------------
# BƯỚC 2: TẠO TỆP NATIONAL CHO SPEC B
# ------------------------------------------------------------
# map mode cho PORT0201
def code_to_mode(code):
    code = int(code)
    if 51 <= code <= 59: return 'Ro-Ro'
    elif 31 <= code <= 34: return 'Lo-Lo'
    return 'Other'

nat = p0201_clean.copy()
nat['Mode'] = nat['CargoCode'].apply(code_to_mode)
nat = nat[nat['Mode'].isin(['Ro-Ro','Lo-Lo'])]
nat = nat[nat['Region'].astype(str).str.contains('International', case=False, na=False)]
nat = nat[nat['Tonnage'] > 0]
national = nat.groupby(['Year','Mode'])['Tonnage'].sum().reset_index()
national = national[(national['Year']>=2015)&(national['Year']<=2024)]
national['log_tonnage'] = np.log(national['Tonnage'])
national['roro'] = (national['Mode']=='Ro-Ro').astype(int)
national['post_brexit'] = (national['Year']>=BREXIT_YEAR).astype(int)
national['covid'] = (national['Year']==COVID_YEAR).astype(int)


# ------------------------------------------------------------
# BƯỚC 3: CHẠY MÔ HÌNH HỒI QUY (SPEC A & SPEC B)
# ------------------------------------------------------------
# ---- Spec A: ALL-port panel ----
a = panel[(panel['Year']>=2015) & (panel['Year']<=2024)].copy()   # panel thay vì focus
cell_means = a.groupby('cell')['Tonnage'].mean()
a = a[a['cell'].isin(cell_means[cell_means >= 1000].index)]
specA = smf.ols('log_tonnage ~ post_brexit * roro + covid + C(cell) + trend',
                data=a).fit(cov_type='HC3')
report(specA, 'Spec A: all-port panel (cells >= 1M tonnes)')

# ---- Spec B: national international difference-in-differences ----
# Spec B từ PORT0201 international only
specB = smf.ols('log_tonnage ~ post_brexit * roro + covid', data=national).fit(cov_type='HC3')
report(specB, 'Spec B: national international DiD')


# ------------------------------------------------------------
# BƯỚC 4: KIỂM ĐỊNH TÍNH VỮNG CHẮC (ROBUSTNESS CHECKS)
# ------------------------------------------------------------
rob_rows = []

def robust(df, label, weights=None):
    """Run one robustness variant with cell-clustered standard errors."""
    kwargs = dict(cov_type='cluster', cov_kwds={'groups': df['cell']})
    formula = 'log_tonnage ~ post_brexit * roro + covid + C(cell) + trend'
    if weights is not None:
        m = smf.wls(formula, data=df, weights=weights).fit(**kwargs)
    else:
        m = smf.ols(formula, data=df).fit(**kwargs)
    
    b, p_ = m.params['post_brexit:roro'], m.pvalues['post_brexit:roro']
    rob_rows.append([label, int(m.nobs), df['cell'].nunique(), 
                     f'{100*(np.exp(b)-1):+.1f}%', round(p_, 4)])

# Lọc các cell lớn
big = panel[panel['cell'].isin(panel.groupby('cell')['Tonnage'].mean().pipe(lambda s: s[s >= 500]).index)].copy()

robust(big, 'P1: all ports, cells >= 0.5M')
robust(big[big['cell'].isin(big.groupby('cell')['Tonnage'].mean().pipe(lambda s: s[s >= 1000]).index)], 
       'P2: all ports, cells >= 1M')
robust(big, 'P3: WLS tonnage-weighted', 
       weights=big.groupby('cell')['Tonnage'].transform('mean'))
robust(big[(big['Year']>=2015)&(big['Year']<=2024)], 'P4: 2015-2024 window')

robustness = pd.DataFrame(rob_rows, columns=['Model', 'n', 'cells', 'BrexitxRoRo', 'p'])
robustness.to_csv(f'{OUT_DIR}/robustness_table.csv', index=False)

print('\nRobustness table (interaction term, cluster SEs):')
print(robustness.to_string(index=False))

with open(f'{OUT_DIR}/regression_full_output.txt', 'w') as f:
    f.write('=== SPEC A ===\n' + str(specA.summary()) 
            + '\n\n=== SPEC B ===\n' + str(specB.summary()))


print("national shape:", national.shape)
print(national)

--- Spec A: all-port panel (cells >= 1M tonnes) | R2=0.974 n=270 ---
         post_brexit: -8.8%  (p=0.0828)
    post_brexit:roro: -1.3%  (p=0.7445)
               covid: -6.7%  (p=0.0498)
--- Spec B: national international DiD | R2=0.983 n=20 ---
         post_brexit: -6.7%  (p=0.0066)
    post_brexit:roro: -21.7%  (p=0.0000)
               covid: -10.6%  (p=0.1735)

Robustness table (interaction term, cluster SEs):
                       Model   n  cells BrexitxRoRo      p
P1: all ports, cells >= 0.5M 573     36       -4.3% 0.7305
  P2: all ports, cells >= 1M 461     29       -0.6% 0.9666
    P3: WLS tonnage-weighted 573     36       -2.3% 0.8835
        P4: 2015-2024 window 334     34       -2.3% 0.7676
national shape: (20, 7)
    Year   Mode        Tonnage  log_tonnage  roro  post_brexit  covid
12  2015  Lo-Lo  118068.678961    11.679022     0            0      0
13  2015  Ro-Ro   90870.828437    11.417194     1            0      0
14  2016  Lo-Lo  123183.774623    11.721433     0 

In [12]:
# ============================================================
# KIỂM TRA p0502_clean NGAY TẠI ĐÂY
# ============================================================
print("Dòng tổng còn không?:", 'Total at all major ports' in p0502_clean['Major Port'].values)
print("Số cảng:", p0502_clean['Major Port'].nunique())
print("Số dòng p0502_clean:", len(p0502_clean))

Dòng tổng còn không?: False
Số cảng: 53
Số dòng p0502_clean: 3604


In [13]:
# ============================================================
# TOTAL QUARTERLY (dữ liệu đã bỏ dòng tổng)
# ============================================================
total_q = (p0502_clean.groupby('Date', as_index=False)['Tonnage'].sum()
           .rename(columns={'Date': 'ds', 'Tonnage': 'y'}))
total_q['ds'] = pd.to_datetime(total_q['ds'])
total_q = total_q.sort_values('ds').reset_index(drop=True)

print("Số quý:", len(total_q))
print("Tổng 4 quý cuối:", round(total_q['y'].tail(4).sum()/1000,1), "triệu tấn")  # phải ~421
print(total_q.tail(6).to_string(index=False))

Số quý: 68
Tổng 4 quý cuối: 419.5 triệu tấn
        ds             y
2024-07-01 102987.590000
2024-10-01 107894.890000
2025-01-01 107546.924340
2025-04-01 106162.667319
2025-07-01 102935.382894
2025-10-01 102894.259536


In [14]:
# Kiểm tra total_q đúng trước khi forecast
print("Kiểm tra: tổng 4 quý cuối =", round(total_q['y'].tail(4).sum()/1000,1), "triệu tấn (phải ~421)")
print("Giá trị 1 quý gần nhất =", round(total_q['y'].iloc[-1]/1000,1), "nghìn... phải ~105")

# ============================================================
# FORECAST TOTAL VOLUME: Prophet vs ARIMA
# ============================================================
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
import itertools

train = total_q[total_q['ds'] <= '2023-10-01'].copy()
test = total_q[total_q['ds'] > '2023-10-01'].copy()
print("Train:", len(train), "quý | Test:", len(test), "quý")

# --- Prophet ---
mp = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
mp.fit(train.rename(columns={'ds':'ds','y':'y'}))
future = mp.make_future_dataframe(periods=len(test), freq='QS')
prophet_pred = mp.predict(future).tail(len(test))['yhat'].values

# --- ARIMA grid search ---
best_aic, best_fit, best_order = np.inf, None, None
for order in itertools.product(range(3),range(2),range(3),range(2),range(2),range(2)):
    p_,d_,q_,P_,D_,Q_ = order
    try:
        r = SARIMAX(train['y'], order=(p_,d_,q_), seasonal_order=(P_,D_,Q_,4),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=0)
        if r.aic < best_aic: best_aic, best_fit, best_order = r.aic, r, (p_,d_,q_,P_,D_,Q_)
    except: continue
if best_fit is None:
    raise ValueError("ARIMA grid search không tìm được model — kiểm tra dữ liệu train")
arima_pred = best_fit.forecast(len(test)).values

# --- Metrics: MAPE, RMSE, MAE ---
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(a,f): return np.mean(np.abs((a-f)/a))*100

actual = test['y'].values

# Tính cả 3 chỉ số cho mỗi mô hình
print("\n=== FORECAST ACCURACY (test 2024-2025) ===")
for name, pred in [('Prophet', prophet_pred), ('ARIMA', arima_pred)]:
    mae_val = mean_absolute_error(actual, pred)
    rmse_val = np.sqrt(mean_squared_error(actual, pred))
    mape_val = mape(actual, pred)
    print(f"{name:8s}  MAE={mae_val:8.1f}  RMSE={rmse_val:8.1f}  MAPE={mape_val:.2f}%")

print(f"\nARIMA order: {best_order}")

# Lưu bảng metrics cho bài
metrics_df = pd.DataFrame([
    {'Model':'Prophet', 
     'MAE': round(mean_absolute_error(actual, prophet_pred),1),
     'RMSE': round(np.sqrt(mean_squared_error(actual, prophet_pred)),1),
     'MAPE': round(mape(actual, prophet_pred),2)},
    {'Model':'ARIMA',
     'MAE': round(mean_absolute_error(actual, arima_pred),1),
     'RMSE': round(np.sqrt(mean_squared_error(actual, arima_pred)),1),
     'MAPE': round(mape(actual, arima_pred),2)},
])
metrics_df.to_csv(f'{OUT_DIR}/forecast_metrics.csv', index=False)
print("\nĐã lưu forecast_metrics.csv:")
print(metrics_df.to_string(index=False))

Kiểm tra: tổng 4 quý cuối = 419.5 triệu tấn (phải ~421)
Giá trị 1 quý gần nhất = 102.9 nghìn... phải ~105
Train: 60 quý | Test: 8 quý


23:18:57 - cmdstanpy - INFO - Chain [1] start processing
23:18:58 - cmdstanpy - INFO - Chain [1] done processing



=== FORECAST ACCURACY (test 2024-2025) ===
Prophet   MAE=  3553.8  RMSE=  4560.2  MAPE=3.33%
ARIMA     MAE=  7581.3  RMSE=  8161.6  MAPE=7.19%

ARIMA order: (2, 1, 2, 1, 1, 1)

Đã lưu forecast_metrics.csv:
  Model    MAE   RMSE  MAPE
Prophet 3553.8 4560.2  3.33
  ARIMA 7581.3 8161.6  7.19


In [15]:
print("Tổng 4 quý cuối:", round(total_q['y'].tail(4).sum()/1000,1), "triệu tấn")
print("1 quý gần nhất:", round(total_q['y'].iloc[-1]/1000,1), "nghìn tấn")

Tổng 4 quý cuối: 419.5 triệu tấn
1 quý gần nhất: 102.9 nghìn tấn


In [16]:
# ============================================================
# FORECAST 2026 (dùng Prophet - model tốt hơn, train toàn bộ)
# ============================================================
# Train lại trên TOÀN BỘ dữ liệu (tới 2025)
mp_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
mp_full.fit(total_q.rename(columns={'ds':'ds','y':'y'}))
future_2026 = mp_full.make_future_dataframe(periods=4, freq='QS')
fc_2026 = mp_full.predict(future_2026).tail(4)[['ds','yhat']]
fc_2026['Quarter'] = ['2026 Q1','2026 Q2','2026 Q3','2026 Q4']
fc_2026['Forecast'] = fc_2026['yhat'].round(1)
print("\nForecast 2026 (Prophet):")
print(fc_2026[['Quarter','Forecast']].to_string(index=False))

# --- Bảng actual vs predicted (test window) cho Table 4.7 ---
compare = test[['ds','y']].copy()
compare['Prophet'] = prophet_pred.round(1)
compare['ARIMA'] = arima_pred.round(1)
compare['Quarter'] = compare['ds'].dt.year.astype(str) + ' Q' + compare['ds'].dt.quarter.astype(str)
compare = compare.rename(columns={'y':'Actual'})
print("\nActual vs Predicted (test window):")
print(compare[['Quarter','Actual','Prophet','ARIMA']].to_string(index=False))

# --- Export CSV cho Power BI ---
compare[['Quarter','Actual','Prophet','ARIMA']].to_csv(f'{OUT_DIR}/forecast_test.csv', index=False)
fc_2026[['Quarter','Forecast']].to_csv(f'{OUT_DIR}/forecast_2026.csv', index=False)
print("\nĐã lưu forecast_test.csv và forecast_2026.csv")

23:19:02 - cmdstanpy - INFO - Chain [1] start processing
23:19:02 - cmdstanpy - INFO - Chain [1] done processing



Forecast 2026 (Prophet):
Quarter  Forecast
2026 Q1  103489.0
2026 Q2  104861.6
2026 Q3  103274.3
2026 Q4  104533.4

Actual vs Predicted (test window):
Quarter        Actual  Prophet    ARIMA
2024 Q1 102476.252000 105734.8 104887.5
2024 Q2 107657.977000 101312.3 100179.8
2024 Q3 102987.590000 102646.1  96613.9
2024 Q4 107894.890000 101147.0 101860.5
2025 Q1 107546.924340  99946.4  98055.4
2025 Q2 106162.667319 102731.0  92727.3
2025 Q3 102935.382894 102446.6  93714.6
2025 Q4 102894.259536 102678.5  96689.2

Đã lưu forecast_test.csv và forecast_2026.csv


In [17]:
# ============================================================
# TẠO DỮ LIỆU FORECAST CHO POWER BI
# ============================================================

# --- 1. Forecast 2026 (Prophet, train toàn bộ tới 2025) ---
mp_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
mp_full.fit(total_q.rename(columns={'ds':'ds','y':'y'}))
future_2026 = mp_full.make_future_dataframe(periods=4, freq='QS')
fc = mp_full.predict(future_2026).tail(4)

# --- 2. Tạo file lịch sử + forecast cho Power BI (dạng long) ---
rows = []

# Phần lịch sử (Actual) - toàn bộ total_q
for _, r in total_q.iterrows():
    d = r['ds']
    rows.append({
        'Period': f"{d.year} Q{(d.month-1)//3+1}",
        'Type': 'Actual',
        'Tonnage': round(r['y'], 1),
        'PeriodOrder': d.year*10 + (d.month-1)//3+1
    })

# Phần forecast 2026 (Forecast)
for _, r in fc.iterrows():
    d = r['ds']
    rows.append({
        'Period': f"{d.year} Q{(d.month-1)//3+1}",
        'Type': 'Forecast',
        'Tonnage': round(r['yhat'], 1),
        'PeriodOrder': d.year*10 + (d.month-1)//3+1
    })

powerbi_forecast = pd.DataFrame(rows).sort_values('PeriodOrder').reset_index(drop=True)
powerbi_forecast.to_csv(f'{OUT_DIR}/powerbi_total_forecast.csv', index=False)
print("Đã lưu powerbi_total_forecast.csv:", powerbi_forecast.shape)
print("\nVài dòng cuối (xem forecast nối tiếp):")
print(powerbi_forecast.tail(8).to_string(index=False))

# --- 3. File so sánh model (test window) cho Power BI ---
compare = test[['ds','y']].copy()
compare['Prophet'] = prophet_pred.round(1)
compare['ARIMA'] = arima_pred.round(1)
compare['Period'] = compare['ds'].dt.year.astype(str) + ' Q' + compare['ds'].dt.quarter.astype(str)
compare = compare.rename(columns={'y':'Actual'})
compare[['Period','Actual','Prophet','ARIMA']].to_csv(f'{OUT_DIR}/powerbi_model_comparison.csv', index=False)
print("\nĐã lưu powerbi_model_comparison.csv")
print(compare[['Period','Actual','Prophet','ARIMA']].to_string(index=False))

23:19:02 - cmdstanpy - INFO - Chain [1] start processing
23:19:02 - cmdstanpy - INFO - Chain [1] done processing


Đã lưu powerbi_total_forecast.csv: (72, 4)

Vài dòng cuối (xem forecast nối tiếp):
 Period     Type  Tonnage  PeriodOrder
2025 Q1   Actual 107546.9        20251
2025 Q2   Actual 106162.7        20252
2025 Q3   Actual 102935.4        20253
2025 Q4   Actual 102894.3        20254
2026 Q1 Forecast 103489.0        20261
2026 Q2 Forecast 104861.6        20262
2026 Q3 Forecast 103274.3        20263
2026 Q4 Forecast 104533.4        20264

Đã lưu powerbi_model_comparison.csv
 Period        Actual  Prophet    ARIMA
2024 Q1 102476.252000 105734.8 104887.5
2024 Q2 107657.977000 101312.3 100179.8
2024 Q3 102987.590000 102646.1  96613.9
2024 Q4 107894.890000 101147.0 101860.5
2025 Q1 107546.924340  99946.4  98055.4
2025 Q2 106162.667319 102731.0  92727.3
2025 Q3 102935.382894 102446.6  93714.6
2025 Q4 102894.259536 102678.5  96689.2


In [18]:
# ============================================================
# TẠO CÁC FILE CSV CHO DASHBOARD POWER BI (theo từng KPI)
# ============================================================

# --- KPI 1: TOTAL VOLUME theo NĂM (từ PORT0301, tất cả cảng) ---
kpi1 = (p0301_clean[
    (p0301_clean['Cargo Group Name'] == 'All Cargo') &
    (p0301_clean['Direction'] == 'Both Directions')]
    .groupby('Year', as_index=False)['Tonnage'].sum())
kpi1['Tonnage_Million'] = (kpi1['Tonnage'] / 1000).round(1)
kpi1.to_csv(f'{OUT_DIR}/kpi1_total_volume.csv', index=False)
print("KPI 1 (total volume theo năm):", kpi1.shape)
print(kpi1[['Year','Tonnage_Million']].to_string(index=False))

# --- KPI 2: Ro-Ro vs Lo-Lo theo NĂM (từ panel) ---
kpi2 = panel.groupby(['Year','Mode'], as_index=False)['Tonnage'].sum()
kpi2['Tonnage_Million'] = (kpi2['Tonnage'] / 1000).round(1)
kpi2.to_csv(f'{OUT_DIR}/kpi2_mode_split.csv', index=False)
print("\nKPI 2 (Ro-Ro vs Lo-Lo theo năm):", kpi2.shape)

# --- KPI 3: Route diversion (từ mode_change, chỉ Ro-Ro) ---
kpi3 = mode_change[mode_change['Mode']=='Ro-Ro'].dropna(subset=['pct_change']).copy()
kpi3.to_csv(f'{OUT_DIR}/kpi3_route_map.csv', index=False)
print("\nKPI 3 (route diversion):", kpi3.shape)

# --- KPI 5: Cargo composition (năm 2024, tất cả nhóm hàng) ---
kpi5 = (p0301_clean[
    (p0301_clean['Direction'] == 'Both Directions') &
    (p0301_clean['Year'] == 2024) &
    (p0301_clean['Cargo Group Name'] != 'All Cargo')]
    .groupby('Cargo Group Name', as_index=False)['Tonnage'].sum())
kpi5['Tonnage_Million'] = (kpi5['Tonnage'] / 1000).round(1)
kpi5['Percentage'] = (kpi5['Tonnage'] / kpi5['Tonnage'].sum() * 100).round(2)
kpi5.to_csv(f'{OUT_DIR}/kpi5_composition.csv', index=False)
print("\nKPI 5 (cargo composition 2024):", kpi5.shape)
print(kpi5[['Cargo Group Name','Percentage']].to_string(index=False))

KPI 1 (total volume theo năm): (17, 3)
 Year  Tonnage_Million
 2009            489.6
 2010            498.5
 2011            507.0
 2012            489.5
 2013            491.8
 2014            491.9
 2015            485.7
 2016            472.8
 2017            470.7
 2018            472.1
 2019            471.7
 2020            429.0
 2021            435.4
 2022            449.7
 2023            425.9
 2024            421.0
 2025            419.5

KPI 2 (Ro-Ro vs Lo-Lo theo năm): (34, 4)

KPI 3 (route diversion): (28, 5)

KPI 5 (cargo composition 2024): (6, 4)
   Cargo Group Name  Percentage
           Dry Bulk       14.07
        Liquid Bulk       28.35
              Lo-Lo       10.55
       Main Freight       26.81
Other General Cargo        2.95
              Ro-Ro       17.26


In [19]:
# Xem route diversion của Lo-Lo (để quyết có nên đưa vào không)
ll = mode_change[mode_change['Mode']=='Lo-Lo'].dropna(subset=['pct_change']).sort_values('pct_change')
print("Lo-Lo — giảm mạnh nhất và tăng mạnh nhất:")
print(pd.concat([ll.head(6), ll.tail(6)])[['PortName','pct_change']].to_string(index=False))

Lo-Lo — giảm mạnh nhất và tăng mạnh nhất:
           PortName  pct_change
         Manchester       -95.4
           Aberdeen       -56.5
            Harwich       -49.8
             Medway       -47.9
        Warrenpoint       -42.2
              Dover       -34.4
  Tees & Hartlepool        -3.4
            Belfast         3.6
Grimsby & Immingham         4.9
            Bristol        10.6
               Hull        15.9
             London        19.2


In [22]:
import numpy as np
print(specB.params)

Intercept           11.736237
post_brexit         -0.069759
roro                -0.366616
post_brexit:roro    -0.244928
covid               -0.111798
dtype: float64


In [23]:
b3 = specB.params['post_brexit:roro']   # sua ten neu khac
print("He so       :", round(b3, 4))
print("So chinh xac:", round((np.exp(b3) - 1) * 100, 2), "%")

He so       : -0.2449
So chinh xac: -21.72 %


In [24]:
print("Spec B - loai sai so chuan:", specB.cov_type)
print("Spec A - loai sai so chuan:", specA.cov_type)

Spec B - loai sai so chuan: HC3
Spec A - loai sai so chuan: HC3


In [21]:
[v for v in dir() if any(k in v.lower() for k in ['spec','res','mod','reg','fit','ols'])]

['__spec__',
 'best_fit',
 'code_to_mode',
 'get_mode',
 'itertools',
 'mode',
 'mode_change',
 'specA',
 'specB']